# Exploration — Localisation & Zone Climatique
Analyse approfondie des groupes géographiques et climatiques avec cartes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_EXTERNAL  = ROOT / 'data' / 'external'
FIGURES        = ROOT / 'reports' / 'figures'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')

CONSO_COL = 'out.electricity.total.energy_consumption..kwh'
CHAUF_COL = 'out.electricity.heating.energy_consumption..kwh'
CLIM_COL  = 'out.electricity.cooling.energy_consumption..kwh'
PIC_ETE   = 'out.qoi.electricity.maximum_daily_peak_summer..kw'
PIC_HIVER = 'out.qoi.electricity.maximum_daily_peak_winter..kw'

for col in [CONSO_COL, CHAUF_COL, CLIM_COL, PIC_ETE, PIC_HIVER]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Shape:', df.shape)

## Localisation — Top états et régions Census

In [ ]:
# Top 20 états : nb bâtiments + conso moy | Répartition par région Census
par_etat = df.groupby('in.state').agg(
    nb=(CONSO_COL, 'count'),
    conso_moy=(CONSO_COL, 'mean')
).sort_values('nb', ascending=False).head(20)

par_region = df.groupby('in.census_region').agg(
    nb=(CONSO_COL, 'count'),
    conso_moy=(CONSO_COL, 'mean')
).sort_values('conso_moy')

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# 1. Nb bâtiments top 20 états (couleur = conso moy)
norm = (par_etat['conso_moy'] - par_etat['conso_moy'].min()) / (par_etat['conso_moy'].max() - par_etat['conso_moy'].min())
colors = plt.cm.RdYlGn_r(norm.values[::-1])
axes[0].barh(par_etat.index[::-1], par_etat['nb'][::-1], color=colors)
axes[0].set_xlabel('Nb batiments')
axes[0].set_title('Top 20 etats — Nb batiments\n(couleur = conso moy : vert=faible, rouge=fort)', fontweight='bold', fontsize=9)

# 2. Conso moy top 20 états
par_conso = par_etat.sort_values('conso_moy')
axes[1].barh(par_conso.index, par_conso['conso_moy'], color='#e74c3c', alpha=0.8)
axes[1].set_xlabel('Conso moy (kWh/an)')
axes[1].set_title('Top 20 etats — Consommation moyenne (kWh/an)', fontweight='bold', fontsize=9)
for bar, val in zip(axes[1].patches, par_conso['conso_moy']):
    axes[1].text(val + 50, bar.get_y() + bar.get_height()/2, f'{val:,.0f}', va='center', fontsize=7)

# 3. Conso moy par région Census
colors_reg = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
bars = axes[2].barh(par_region.index, par_region['conso_moy'], color=colors_reg, alpha=0.85)
axes[2].set_xlabel('Conso moy (kWh/an)')
axes[2].set_title('Consommation moyenne par region Census', fontweight='bold', fontsize=9)
for bar, (_, row) in zip(axes[2].patches, par_region.iterrows()):
    axes[2].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                 f"{row['nb']:,} batiments", va='center', fontsize=8)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Localisation — Repartition et consommation electrique par etat et region', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'localisation_etats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegarde : localisation_etats.png')

In [ ]:
# Charger le geojson des etats
gdf = gpd.read_file(DATA_EXTERNAL / 'map_of_us_states.geojson')

# Compter les batiments par etat
counts = df['in.state'].value_counts().reset_index()
counts.columns = ['state_abbr', 'nb_batiments']

# Joindre avec le geojson sur l'abréviation de l'état
gdf = gdf.merge(counts, left_on='iso_3166_2', right_on='state_abbr', how='left')

fig, ax = plt.subplots(figsize=(14, 8))
gdf.plot(
    column='nb_batiments',
    cmap='Blues',
    linewidth=0.5,
    edgecolor='gray',
    legend=True,
    ax=ax,
    missing_kwds={'color': 'lightgray'}
)
ax.set_title('Nombre de batiments par etat (ResStock)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES / 'carte_batiments_par_etat.png', dpi=150, bbox_inches='tight')
plt.show()

## Zone climatique ASHRAE — Distribution, consommation et pics de puissance

In [ ]:
zones_order = sorted(df['in.ashrae_iecc_climate_zone_2004'].dropna().unique())

grp = df.groupby('in.ashrae_iecc_climate_zone_2004').agg(
    nb=(CONSO_COL, 'count'),
    chauffage=(CHAUF_COL, 'mean'),
    clim=(CLIM_COL, 'mean'),
    pic_ete=(PIC_ETE, 'mean'),
    pic_hiver=(PIC_HIVER, 'mean'),
).reindex(zones_order)

x = list(range(len(zones_order)))
w = 0.38

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# 1. Nb bâtiments par zone
axes[0].bar(x, grp['nb'], color='#5b9bd5', alpha=0.85, edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(zones_order, rotation=45, fontsize=9)
axes[0].set_ylabel('Nb batiments'); axes[0].set_title('Nb batiments par zone', fontweight='bold')
for i, val in enumerate(grp['nb']):
    axes[0].text(i, val + 500, f'{int(val):,}', ha='center', fontsize=7)

# 2. Chauffage vs Climatisation
axes[1].bar([i - w/2 for i in x], grp['chauffage'], width=w, label='Chauffage', color='#f46d43', alpha=0.9)
axes[1].bar([i + w/2 for i in x], grp['clim'],      width=w, label='Climatisation', color='#74add1', alpha=0.9)
axes[1].set_xticks(x); axes[1].set_xticklabels(zones_order, rotation=45, fontsize=9)
axes[1].set_ylabel('kWh/an'); axes[1].set_title('Chauffage vs Climatisation (kWh/an)', fontweight='bold')
axes[1].legend(fontsize=9)

# 3. Pics de puissance (directement utile FlexiMax)
axes[2].bar([i - w/2 for i in x], grp['pic_ete'],   width=w, label='Pic ete', color='#d73027', alpha=0.9)
axes[2].bar([i + w/2 for i in x], grp['pic_hiver'], width=w, label='Pic hiver', color='#4575b4', alpha=0.9)
axes[2].set_xticks(x); axes[2].set_xticklabels(zones_order, rotation=45, fontsize=9)
axes[2].set_ylabel('kW'); axes[2].set_title('Pic de puissance ete vs hiver (kW)\n[cle FlexiMax]', fontweight='bold')
axes[2].legend(fontsize=9)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Zone climatique ASHRAE — Distribution, usage energetique et flexibilite', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'zones_climatiques_synthese.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegarde : zones_climatiques_synthese.png')

In [ ]:
# Zone climatique dominante par etat
zone_par_etat = (
    df.groupby('in.state')['in.ashrae_iecc_climate_zone_2004']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
zone_par_etat.columns = ['state_abbr', 'zone_dominante']
zone_par_etat['zone_dominante'] = zone_par_etat['zone_dominante'].astype(str)

gdf2 = gpd.read_file(DATA_EXTERNAL / 'map_of_us_states.geojson')
gdf2 = gdf2.merge(zone_par_etat, left_on='iso_3166_2', right_on='state_abbr', how='left')

# Calculer zone_num apres le merge — extraire le 1er chiffre ('1A'->'1', '7AK'->'7')
gdf2['zone_num'] = gdf2['zone_dominante'].astype(str).str.extract(r'(\d)')[0]

# Palette par numero de zone (1=tres chaud rouge -> 8=arctique bleu fonce)
palette = {
    '1': '#d73027', '2': '#f46d43', '3': '#fdae61',
    '4': '#fee090', '5': '#abd9e9', '6': '#74add1',
    '7': '#4575b4', '8': '#313695'
}
gdf2['couleur'] = gdf2['zone_num'].map(palette).fillna('lightgray')

fig, ax = plt.subplots(figsize=(14, 8))
gdf2.plot(color=gdf2['couleur'], linewidth=0.5, edgecolor='white', ax=ax)

from matplotlib.patches import Patch
labels_legende = [
    ('1', '#d73027', 'Zone 1 — Tres chaud'),
    ('2', '#f46d43', 'Zone 2 — Chaud'),
    ('3', '#fdae61', 'Zone 3 — Chaud tempere'),
    ('4', '#fee090', 'Zone 4 — Mixte'),
    ('5', '#abd9e9', 'Zone 5 — Froid'),
    ('6', '#74add1', 'Zone 6 — Tres froid'),
    ('7', '#4575b4', 'Zone 7 — Subarctique'),
    ('8', '#313695', 'Zone 8 — Arctique'),
]
patches = [Patch(color=c, label=l) for _, c, l in labels_legende]
ax.legend(handles=patches, loc='lower left', fontsize=9)
ax.set_title('Zone climatique ASHRAE dominante par etat', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES / 'carte_zones_climatiques.png', dpi=150, bbox_inches='tight')
plt.show()